# Bai tap
### Thiet ke Auto Trade, chay 1p chay 1 lan - moi khi chay thi se quet du lieu Forex/ Crypto/ Chung khoan theo Timeframe 1m
### Cho ra tin hieu Buy/ Sell: MA10 > MA20 thi Buy, nguoc lai Sell

# Buoc 1: Load data

In [ ]:
def loaddataCrypto(symbol, timeframe): # Dua 2 tham so vao: symbol, timeframe
	import ccxt # ccxt hoac binance
	import pandas as pd

	# Khởi tạo sàn giao dịch Binance
	exchange = ccxt.binance()

	# Đặt cặp giao dịch (BTC/USDT) và khoảng thời gian (H1 là mỗi giờ)

	# Lấy lịch sử giá
	ohlcv = exchange.fetch_ohlcv(symbol, timeframe, limit=1000) # Ham/ EndPoint fetch_ohlcv

	# Chuyển dữ liệu thành DataFrame
	data = pd.DataFrame(ohlcv, columns=['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume'])

	# Chuyển đổi timestamp sang dạng ngày tháng
	data['Datetime'] = pd.to_datetime(data['Datetime'], unit='ms')

	return data

# Test

In [ ]:
# # ##############################################Step 0: Định nghĩa các tham số##############################################
# symbol = 'ETHUSDT'
# timeframe = '1m'

# # ##############################################Step 1: Lấy dữ liệu##############################################
# data = loaddataCrypto(symbol, timeframe)

# data

# Buoc 2 Thiet ke cho chay tu dong 1m chay 1 lan => Khi chay thi goi Load data o tren => Voi data tra ve, thi tinh MA10, MA20 => Sau do tinh Buy_Signal/ Sell_Signal

In [3]:
from datetime import datetime, timedelta
import schedule
import time
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import talib
import redis 

def scan_market():
	# # ##############################################Step 0: Định nghĩa các tham số##############################################
	print('Bắt đầu quét thị trường...')
	print('Thời gian hiện tại:', datetime.now().isoformat())

	symbol = 'C98USDT'
	timeframe = '1m'

	# ##############################################Step 1: Lấy dữ liệu###########################################################
	data = loaddataCrypto(symbol, timeframe)

	# ##############################################Step 2: Tinh toán chiến lược Buy_Signal, Sell_Signal##########################
	data['MA10'] = talib.SMA(data['Close'], timeperiod=10)
	data['MA20'] = talib.SMA(data['Close'], timeperiod=20)

	# Buy Signal/ Sell Signal
	data['Buy_Signal'] = (data['MA10'] > data['MA20'])
	data['Sell_Signal'] = (data['MA10'] < data['MA20'])

	# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
	# Nếu có tín hiệu thì đẩy qua Redis
	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=15) # DB muon chua: 0->5 CK, 6->10 FX, 11->15 Crypto
	# Tạo hash key
	hash_key = 'OG_CRTO_MA10, MA20_' + symbol
	last_record = data.iloc[-1].copy() # Lay record moi nhat
   
	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True): # Neu co tin hieu Buy hoặc Sell
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
			last_record['Insertdate'] = datetime.now().isoformat()
		print(last_record)   
	else:
		print(last_record)   
		print('Không có tín hiệu!')

	print('Quét thị trường hoàn tất!')
	print('Thời gian kết thúc:', datetime.now().isoformat())

In [ ]:
from datetime import datetime, timedelta
import schedule
import time
import random

# Danh sách các phút cụ thể bạn muốn hàm được chạy
run_minutes = list(range(0, 60, 1)) # 0,1,2,3,4...59 : Cac phut chung ta can chay

# Thiết lập thời gian chạy cuối cùng để theo dõi
last_run = None
setattr(scan_market, 'last_run', None)

while True: # Luon luon kiem tra
	current_time = datetime.now() # Lay thoi gian hien tai 
	current_minute = current_time.minute # Lay phut hien
	
	# Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
	if current_minute in run_minutes:
		time.sleep(5) # Co the kiem tra o giay thu 5 moi chay hoac time.sleep(5)
		
		# Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
		last_run = getattr(scan_market, 'last_run', None)
		if last_run is None or last_run != current_minute:
			scan_market() # Cho chay
			# Lưu lại phút cuối cùng mà hàm đã chạy
			setattr(scan_market, 'last_run', current_minute)

Bắt đầu quét thị trường...
Thời gian hiện tại: 2025-06-18T22:14:11.770100
Datetime              2025-06-18 15:14:00
Open                               0.0459
High                               0.0459
Low                                0.0459
Close                              0.0459
Volume                              139.2
MA10                              0.04611
MA20                              0.04613
Buy_Signal                          False
Sell_Signal                          True
Insertdate     2025-06-18T22:14:33.652942
Name: 999, dtype: object
Quét thị trường hoàn tất!
Thời gian kết thúc: 2025-06-18T22:14:33.657474
Bắt đầu quét thị trường...
Thời gian hiện tại: 2025-06-18T22:15:08.662111
Datetime              2025-06-18 15:15:00
Open                               0.0459
High                               0.0459
Low                                0.0459
Close                              0.0459
Volume                              139.2
MA10                              0.0460